In [1]:
!pip install scikit-surprise -q

import pandas as pd
import numpy as np
import pickle, time
from scipy import sparse
from surprise import Dataset, Reader, SVD
import torch
import torch.nn as nn

print("CUDA available:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print("using device:", device)

# Adjust these to match your uploaded Kaggle datasets
INTERIM_PATH = "/kaggle/input/datasets/anurajgogoi/cineiq-interim/interim"
SVD_PATH = "/kaggle/input/datasets/anurajgogoi/cineiq-interim/interim/svd"         
CONTENT_PATH = "/kaggle/input/datasets/anurajgogoi/cineiq-interim/interim"  
GRU_PATH = "/kaggle/input/datasets/anurajgogoi/cineiq-interim/interim"     

# --- SVD ---
with open(f"{SVD_PATH}/svd_model.pkl", "rb") as f:
    svd_model = pickle.load(f)
print("SVD loaded")

# --- Content (TF-IDF) ---
with open(f"{CONTENT_PATH}/tfidf_vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)
tfidf_matrix = sparse.load_npz(f"{CONTENT_PATH}/tfidf_matrix.npz")
movies_master = pd.read_parquet(f"{INTERIM_PATH}/movies_master.parquet")
print("content model loaded, tfidf_matrix shape:", tfidf_matrix.shape)

# --- GRU ---
class GRU4RecTied(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, hidden_dim=128, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding_dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(embedding_dim, hidden_dim, num_layers=1, batch_first=True)
        self.output_dropout = nn.Dropout(dropout)
        self.projection = nn.Identity() if hidden_dim == embedding_dim else nn.Linear(hidden_dim, embedding_dim)
        self.output_bias = nn.Parameter(torch.zeros(vocab_size))

    def embed_sequence(self, input_seqs, lengths):
        embedded = self.embedding_dropout(self.embedding(input_seqs))
        packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, hidden = self.gru(packed)
        return self.projection(self.output_dropout(hidden[-1]))

    def score_full_vocab(self, projected):
        return projected @ self.embedding.weight.T + self.output_bias

    def forward(self, input_seqs, lengths):
        return self.score_full_vocab(self.embed_sequence(input_seqs, lengths))


with open(f"{GRU_PATH}/gru_mappings.pkl", "rb") as f:
    gru_meta = pickle.load(f)
gru_movie_to_idx = gru_meta["movie_to_idx"]
gru_idx_to_movie = gru_meta["idx_to_movie"]
gru_vocab_size = len(gru_movie_to_idx) + 1

gru_model = GRU4RecTied(vocab_size=gru_vocab_size, embedding_dim=64, hidden_dim=128, dropout=0.3)
gru_model.load_state_dict(torch.load(f"{GRU_PATH}/gru_model_final.pt", map_location=device))
gru_model.to(device)
gru_model.eval()
print("GRU loaded, vocab size:", gru_vocab_size)

print("\nall 3 models loaded successfully")

CUDA available: True
using device: cuda
SVD loaded


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


content model loaded, tfidf_matrix shape: (62423, 35949)
GRU loaded, vocab size: 37335

all 3 models loaded successfully


In [2]:
# CELL 2 — Load ratings, build labeled examples for ALL warm users (not just a dev subsample)

ratings_val_warm = pd.read_parquet(f"{INTERIM_PATH}/ratings_val_warm.parquet")
ratings_train = pd.read_parquet(f"{INTERIM_PATH}/ratings_train.parquet",
                                 columns=["userId", "movieId", "rating", "timestamp"])
warm_users = set(pd.read_parquet(f"{INTERIM_PATH}/warm_users.parquet")["userId"])

print("ratings_val_warm shape:", ratings_val_warm.shape)
print("warm users:", len(warm_users))

def build_labeled_examples(ratings_val_warm, all_movie_ids, positive_threshold=4.0,
                            n_random_negatives_per_user=5, random_state=42):
    rng = np.random.RandomState(random_state)

    real_examples = ratings_val_warm[["userId", "movieId", "rating"]].copy()
    real_examples["label"] = (real_examples["rating"] >= positive_threshold).astype(int)
    real_examples = real_examples[["userId", "movieId", "label"]]

    user_seen = ratings_val_warm.groupby("userId")["movieId"].apply(set).to_dict()

    negative_rows = []
    all_movie_ids_arr = np.array(all_movie_ids)
    for user_id, seen_movies in user_seen.items():
        candidates = rng.choice(all_movie_ids_arr, size=n_random_negatives_per_user * 3, replace=False)
        chosen = [m for m in candidates if m not in seen_movies][:n_random_negatives_per_user]
        for movie_id in chosen:
            negative_rows.append({"userId": user_id, "movieId": movie_id, "label": 0})

    random_negatives = pd.DataFrame(negative_rows)
    combined = pd.concat([real_examples, random_negatives], ignore_index=True)
    return combined


all_movie_ids = movies_master["movieId"].tolist()

start = time.time()
examples = build_labeled_examples(ratings_val_warm, all_movie_ids,
                                   positive_threshold=4.0, n_random_negatives_per_user=5)
print(f"built {len(examples)} examples in {time.time()-start:.1f}s")
print("label distribution:")
print(examples["label"].value_counts())

ratings_val_warm shape: (412348, 6)
warm users: 126591
built 439618 examples in 5.9s
label distribution:
label
0    276010
1    163608
Name: count, dtype: int64


In [3]:
# CELL 3 — Score every example with SVD, content, and GRU

def build_user_train_sequences(ratings_train, movie_to_idx, user_col="userId", item_col="movieId",
                                timestamp_col="timestamp", max_seq_len=100):
    sorted_ratings = ratings_train.sort_values([user_col, timestamp_col])
    sequences = {}
    for user_id, group in sorted_ratings.groupby(user_col):
        encoded = [movie_to_idx[m] for m in group[item_col].tolist() if m in movie_to_idx]
        if len(encoded) > max_seq_len:
            encoded = encoded[-max_seq_len:]
        sequences[user_id] = encoded
    return sequences


def score_candidates_gru(model, user_sequence, candidate_movie_ids, movie_to_idx, device="cpu"):
    model.eval()
    with torch.no_grad():
        input_tensor = torch.tensor([user_sequence], dtype=torch.long).to(device)
        length_tensor = torch.tensor([len(user_sequence)])
        logits = model(input_tensor, length_tensor)[0]
    scores = []
    for movie_id in candidate_movie_ids:
        scores.append(logits[movie_to_idx[movie_id]].item() if movie_id in movie_to_idx else float("-inf"))
    return pd.DataFrame({"movieId": candidate_movie_ids, "gru_score": scores})


def score_candidates_content(user_liked_movie_ids, candidate_movie_ids, movies_master, tfidf_matrix,
                              min_content_tokens=6):
    from sklearn.metrics.pairwise import cosine_similarity
    id_to_idx = {mid: i for i, mid in enumerate(movies_master["movieId"])}
    token_counts = movies_master["content_text"].str.split().str.len().fillna(0).values

    liked_idxs = [id_to_idx[m] for m in user_liked_movie_ids if m in id_to_idx]
    liked_idxs = [i for i in liked_idxs if token_counts[i] >= min_content_tokens]

    if not liked_idxs:
        return pd.DataFrame({"movieId": candidate_movie_ids, "content_score": 0.0})

    valid_candidates = [m for m in candidate_movie_ids if m in id_to_idx]
    candidate_idxs = [id_to_idx[m] for m in valid_candidates]
    sim_matrix = cosine_similarity(tfidf_matrix[candidate_idxs], tfidf_matrix[liked_idxs])
    avg_sims = sim_matrix.mean(axis=1)
    candidate_token_counts = token_counts[candidate_idxs]
    avg_sims = np.where(candidate_token_counts < min_content_tokens, 0.0, avg_sims)
    return pd.DataFrame({"movieId": valid_candidates, "content_score": avg_sims})


def score_batch_svd(model, user_item_pairs, user_col="userId", item_col="movieId"):
    out = user_item_pairs.copy()
    out["svd_score"] = out.apply(lambda row: model.predict(row[user_col], row[item_col]).est, axis=1)
    return out


# Restrict train data to just the val-warm users being scored, for speed/memory
ratings_train_relevant = ratings_train[ratings_train["userId"].isin(examples["userId"].unique())]

print("building GRU sequences...")
user_train_sequences = build_user_train_sequences(ratings_train_relevant, gru_movie_to_idx)

print("building content liked-movies lookup...")
ratings_train_by_user = (
    ratings_train_relevant[ratings_train_relevant["rating"] >= 4.0]
    .groupby("userId")["movieId"].apply(list).to_dict()
)

print("scoring all examples (this is the slow part)...")
scored_chunks = []
start = time.time()
n_users = examples["userId"].nunique()

for i, (user_id, group) in enumerate(examples.groupby("userId")):
    candidate_movie_ids = group["movieId"].tolist()

    svd_pairs = pd.DataFrame({"userId": user_id, "movieId": candidate_movie_ids})
    svd_scored = score_batch_svd(svd_model, svd_pairs)

    liked_movies = ratings_train_by_user.get(user_id, [])
    content_scored = score_candidates_content(liked_movies, candidate_movie_ids, movies_master, tfidf_matrix)

    user_seq = user_train_sequences.get(user_id, [])
    if len(user_seq) >= 1:
        gru_scored = score_candidates_gru(gru_model, user_seq, candidate_movie_ids, gru_movie_to_idx, device=device)
    else:
        gru_scored = pd.DataFrame({"movieId": candidate_movie_ids, "gru_score": 0.0})

    merged = group.merge(svd_scored[["movieId", "svd_score"]], on="movieId", how="left")
    merged = merged.merge(content_scored[["movieId", "content_score"]], on="movieId", how="left")
    merged = merged.merge(gru_scored[["movieId", "gru_score"]], on="movieId", how="left")
    scored_chunks.append(merged)

    if i % 500 == 0:
        elapsed = time.time() - start
        print(f"  {i}/{n_users} users scored — {elapsed:.1f}s elapsed")

scored_examples = pd.concat(scored_chunks, ignore_index=True)
print(f"\ntotal scoring time: {time.time()-start:.1f}s")
print("scored_examples shape:", scored_examples.shape)
print(scored_examples.head(10))
print("\nnull check:")
print(scored_examples[["svd_score", "content_score", "gru_score"]].isna().sum())

building GRU sequences...
building content liked-movies lookup...
scoring all examples (this is the slow part)...
  0/5454 users scored — 0.6s elapsed
  500/5454 users scored — 229.0s elapsed
  1000/5454 users scored — 456.4s elapsed
  1500/5454 users scored — 681.8s elapsed
  2000/5454 users scored — 902.1s elapsed
  2500/5454 users scored — 1119.9s elapsed
  3000/5454 users scored — 1335.7s elapsed
  3500/5454 users scored — 1548.0s elapsed
  4000/5454 users scored — 1761.0s elapsed
  4500/5454 users scored — 1974.1s elapsed
  5000/5454 users scored — 2184.9s elapsed

total scoring time: 2373.1s
scored_examples shape: (439618, 6)
   userId  movieId  label  svd_score  content_score  gru_score
0       3       29      1   4.317312       0.021638   0.964769
1       3      111      1   4.427020       0.006722   4.059372
2       3      214      1   4.455537       0.003852  -0.376699
3       3      293      1   4.273249       0.014818   4.077610
4       3      741      1   4.559973       0.

In [4]:
print("Score distributions by label")
print("\nSVD score:")
print(scored_examples.groupby("label")["svd_score"].describe())

print("\ncontent_score:")
print(scored_examples.groupby("label")["content_score"].describe())

print("\ngru_score:")
print(scored_examples.groupby("label")["gru_score"].describe())

# Save
scored_examples.to_parquet("/kaggle/working/meta_model_training_data.parquet", index=False)
print("\nsaved meta_model_training_data.parquet")

Score distributions by label

SVD score:
          count      mean       std  min       25%       50%      75%  max
label                                                                     
0      276010.0  3.212963  0.634082  0.5  2.897999  3.284475  3.62836  5.0
1      163608.0  3.824734  0.497452  0.5  3.522182  3.851933  4.16192  5.0

content_score:
          count      mean       std  min       25%       50%       75%  \
label                                                                    
0      276010.0  0.008502  0.006354  0.0  0.004459  0.007749  0.011499   
1      163608.0  0.009917  0.007635  0.0  0.005449  0.008662  0.012779   

            max  
label            
0      0.170117  
1      0.152871  

gru_score:
          count  mean  std  min       25%       50%       75%        max
label                                                                   
0      276010.0  -inf  NaN -inf -1.277896  0.893584  2.885075  12.765740
1      163608.0  -inf  NaN -inf -0.128867  

In [5]:
# CELL 5 — Diagnose -inf in gru_score, and check content_score's zero-rate

n_inf = np.isinf(scored_examples["gru_score"]).sum()
print(f"gru_score == -inf: {n_inf} ({100*n_inf/len(scored_examples):.2f}%)")
print("\nby label:")
print(scored_examples.groupby("label").apply(lambda g: np.isinf(g["gru_score"]).sum()))

n_zero_content = (scored_examples["content_score"] == 0).sum()
print(f"\ncontent_score == 0: {n_zero_content} ({100*n_zero_content/len(scored_examples):.2f}%)")
print("by label:")
print(scored_examples.groupby("label").apply(lambda g: (g["content_score"] == 0).sum()))

gru_score == -inf: 62784 (14.28%)

by label:
label
0    43079
1    19705
dtype: int64

content_score == 0: 45168 (10.27%)
by label:
label
0    31061
1    14107
dtype: int64


/tmp/ipykernel_58/4254925904.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(scored_examples.groupby("label").apply(lambda g: np.isinf(g["gru_score"]).sum()))
/tmp/ipykernel_58/4254925904.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(scored_examples.groupby("label").apply(lambda g: (g["content_score"] == 0).sum()))


In [6]:
# CELL 6 — Replace -inf with a finite sentinel value, re-verify distributions

# Use the minimum REAL (non-inf) gru_score as the sentinel -- this preserves
# the meaning "GRU has no real opinion here" as a low-but-finite value,
# rather than an artificial choice disconnected from the score's actual scale
finite_scores = scored_examples.loc[~np.isinf(scored_examples["gru_score"]), "gru_score"]
sentinel_value = finite_scores.min()
print(f"sentinel value (min real gru_score): {sentinel_value:.4f}")

scored_examples["gru_score"] = scored_examples["gru_score"].replace(-np.inf, sentinel_value)

print("\ngru_score after fix:")
print(scored_examples.groupby("label")["gru_score"].describe())

# content_score zero-rate: is this consistent with Week 2's known ~40% exclusion rate,
# or is something else going on? 10.27% here is lower than 40% because that 40% was
# CATALOG-wide -- these are specific candidate movies actually being tested, likely
# skewed toward more mainstream/popular titles (which tend to have richer metadata)
print(f"\ncontent_score == 0 rate: {100*n_zero_content/len(scored_examples):.2f}% "
      f"(vs ~40% catalog-wide exclusion rate from Week 2 -- lower here is expected, "
      f"since these candidates skew toward more mainstream/popular movies)")

# Save the corrected dataset
scored_examples.to_parquet("/kaggle/working/meta_model_training_data.parquet", index=False)
print("\nre-saved corrected meta_model_training_data.parquet")

sentinel value (min real gru_score): -9.0396

gru_score after fix:
          count      mean       std       min       25%       50%       75%  \
label                                                                         
0      276010.0 -0.051233  4.419182 -9.039619 -1.277896  0.893584  2.885075   
1      163608.0  1.224461  4.411427 -9.039619 -0.128867  2.220681  4.036139   

             max  
label             
0      12.765740  
1      15.046813  

content_score == 0 rate: 10.27% (vs ~40% catalog-wide exclusion rate from Week 2 -- lower here is expected, since these candidates skew toward more mainstream/popular movies)

re-saved corrected meta_model_training_data.parquet


In [8]:
# CELL 7 — Train the meta-model

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler

feature_cols = ["svd_score", "content_score", "gru_score"]
X = scored_examples[feature_cols]
y = scored_examples["label"]

# Split the meta-model's OWN train/test (separate from the outer MovieLens
# temporal split -- this is just for evaluating the meta-model's fit quality)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Standardize features -- the 3 signals are on very different scales
# (SVD ~0.5-5, content ~0-0.17, GRU ~-9 to 15), and logistic regression's
# coefficients are only directly comparable/interpretable post-scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

meta_model = LogisticRegression(random_state=42, max_iter=1000)
meta_model.fit(X_train_scaled, y_train)

train_auc = roc_auc_score(y_train, meta_model.predict_proba(X_train_scaled)[:, 1])
test_auc = roc_auc_score(y_test, meta_model.predict_proba(X_test_scaled)[:, 1])

print(f"train AUC: {train_auc:.4f}")
print(f"test AUC: {test_auc:.4f}")

print("\ntest classification report:")
print(classification_report(y_test, meta_model.predict(X_test_scaled)))

print("\nlearned coefficients:")
for name, coef in zip(feature_cols, meta_model.coef_[0]):
    print(f"  {name}: {coef:.4f}")
print(f"  intercept: {meta_model.intercept_[0]:.4f}")

train AUC: 0.7881
test AUC: 0.7877

test classification report:
              precision    recall  f1-score   support

           0       0.76      0.85      0.80     55202
           1       0.68      0.55      0.61     32722

    accuracy                           0.74     87924
   macro avg       0.72      0.70      0.71     87924
weighted avg       0.73      0.74      0.73     87924


learned coefficients:
  svd_score: 1.3708
  content_score: 0.0996
  gru_score: 0.0494
  intercept: -0.7459


In [9]:
print("feature correlations:")
print(scored_examples[feature_cols].corr())

feature correlations:
               svd_score  content_score  gru_score
svd_score       1.000000       0.105955   0.169268
content_score   0.105955       1.000000   0.350320
gru_score       0.169268       0.350320   1.000000


In [10]:
# GBM (GradientBoostingClassifier)

from sklearn.ensemble import GradientBoostingClassifier

gbm_model = GradientBoostingClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42
)
gbm_model.fit(X_train, y_train)  # GBMs don't need standardized features

gbm_train_auc = roc_auc_score(y_train, gbm_model.predict_proba(X_train)[:, 1])
gbm_test_auc = roc_auc_score(y_test, gbm_model.predict_proba(X_test)[:, 1])

print(f"GBM train AUC: {gbm_train_auc:.4f}")
print(f"GBM test AUC: {gbm_test_auc:.4f}")

print(f"\nlogistic regression test AUC: {test_auc:.4f}  (for comparison)")

print("\nGBM feature importances:")
for name, imp in zip(feature_cols, gbm_model.feature_importances_):
    print(f"  {name}: {imp:.4f}")

print("\nGBM classification report:")
print(classification_report(y_test, gbm_model.predict(X_test)))

GBM train AUC: 0.7944
GBM test AUC: 0.7929

logistic regression test AUC: 0.7877  (for comparison)

GBM feature importances:
  svd_score: 0.9478
  content_score: 0.0112
  gru_score: 0.0409

GBM classification report:
              precision    recall  f1-score   support

           0       0.76      0.86      0.81     55202
           1       0.69      0.54      0.61     32722

    accuracy                           0.74     87924
   macro avg       0.73      0.70      0.71     87924
weighted avg       0.73      0.74      0.73     87924



In [11]:
# Save the final meta-model, scaler, and results summary

import pickle, json

with open("/kaggle/working/meta_model.pkl", "wb") as f:
    pickle.dump(meta_model, f)

with open("/kaggle/working/meta_model_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

meta_model_results = {
    "model_type": "Logistic Regression",
    "features": feature_cols,
    "train_auc": round(train_auc, 4),
    "test_auc": round(test_auc, 4),
    "coefficients": {name: round(float(coef), 4) for name, coef in zip(feature_cols, meta_model.coef_[0])},
    "intercept": round(float(meta_model.intercept_[0]), 4),
    "gbm_comparison": {
        "test_auc": round(gbm_test_auc, 4),
        "feature_importances": {name: round(float(imp), 4) for name, imp in zip(feature_cols, gbm_model.feature_importances_)},
        "note": "GBM AUC was marginally higher (0.7929 vs 0.7877) but concentrated "
                "~95% of importance on SVD alone. Logistic regression kept as final "
                "for clearer, more balanced signal-contribution interpretability "
                "(Section 5.6/7 explainability requirement).",
    },
    "feature_correlation": scored_examples[feature_cols].corr().round(4).to_dict(),
    "finding": (
        "SVD is the dominant signal in the learned blend. GRU shows a larger "
        "raw mean gap between positive/negative examples than SVD in absolute "
        "terms, but a much higher variance, so its STANDARDIZED contribution "
        "is smaller -- SVD provides a cleaner, lower-noise signal for this task. "
        "Content-based similarity contributes the least, consistent with Week 2's "
        "finding that ~40% of the catalog has insufficient metadata for reliable "
        "content scoring."
    ),
    "training_data_size": len(scored_examples),
    "training_data_users": int(scored_examples["userId"].nunique()),
    "gru_score_sentinel_value": round(float(sentinel_value), 4),
}

with open("/kaggle/working/meta_model_results.json", "w") as f:
    json.dump(meta_model_results, f, indent=2)

print(json.dumps({k: v for k, v in meta_model_results.items() if k not in ["feature_correlation"]}, indent=2))
print("\nsaved: meta_model.pkl, meta_model_scaler.pkl, meta_model_results.json")

{
  "model_type": "Logistic Regression",
  "features": [
    "svd_score",
    "content_score",
    "gru_score"
  ],
  "train_auc": 0.7881,
  "test_auc": 0.7877,
  "coefficients": {
    "svd_score": 1.3708,
    "content_score": 0.0996,
    "gru_score": 0.0494
  },
  "intercept": -0.7459,
  "gbm_comparison": {
    "test_auc": 0.7929,
    "feature_importances": {
      "svd_score": 0.9478,
      "content_score": 0.0112,
      "gru_score": 0.0409
    },
    "note": "GBM AUC was marginally higher (0.7929 vs 0.7877) but concentrated ~95% of importance on SVD alone. Logistic regression kept as final for clearer, more balanced signal-contribution interpretability (Section 5.6/7 explainability requirement)."
  },
  "finding": "SVD is the dominant signal in the learned blend. GRU shows a larger raw mean gap between positive/negative examples than SVD in absolute terms, but a much higher variance, so its STANDARDIZED contribution is smaller -- SVD provides a cleaner, lower-noise signal for this t

In [13]:
# CELL 11 — Full Top-N generation: score candidates -> meta-model blend -> ranked list

def generate_top_n(user_id, ratings_train_relevant, user_train_sequences, ratings_train_by_user,
                    svd_model, tfidf_matrix, movies_master, gru_model, gru_movie_to_idx,
                    meta_model, scaler, candidate_movie_ids=None, top_n=10, device="cpu"):
    """
    Full meta-model pipeline for one user: score a candidate pool with
    SVD + content + GRU, blend via the trained meta-model, return the
    top_n ranked movieIds with their blended probability.
    """
    if candidate_movie_ids is None:
        # Reasonable default candidate pool: movies in the GRU's vocabulary
        # (i.e. seen during train) minus whatever the user already watched
        already_seen = set(ratings_train_relevant[ratings_train_relevant["userId"] == user_id]["movieId"])
        candidate_movie_ids = [m for m in gru_movie_to_idx.keys() if m not in already_seen]

    svd_pairs = pd.DataFrame({"userId": user_id, "movieId": candidate_movie_ids})
    svd_scored = score_batch_svd(svd_model, svd_pairs)

    liked_movies = ratings_train_by_user.get(user_id, [])
    content_scored = score_candidates_content(liked_movies, candidate_movie_ids, movies_master, tfidf_matrix)

    user_seq = user_train_sequences.get(user_id, [])
    gru_scored = score_candidates_gru(gru_model, user_seq, candidate_movie_ids, gru_movie_to_idx, device=device)

    merged = svd_scored[["movieId", "svd_score"]].merge(
        content_scored[["movieId", "content_score"]], on="movieId", how="inner"
    ).merge(gru_scored[["movieId", "gru_score"]], on="movieId", how="inner")

    merged["gru_score"] = merged["gru_score"].replace(-np.inf, sentinel_value)

    X = scaler.transform(merged[feature_cols])
    merged["meta_score"] = meta_model.predict_proba(X)[:, 1]

    top_n_result = merged.sort_values("meta_score", ascending=False).head(top_n)
    return top_n_result.merge(movies_master[["movieId", "title"]], on="movieId")


# Test on a real warm user
test_user = list(user_train_sequences.keys())[0]
print(f"generating Top-10 for userId={test_user}")

top_n_result = generate_top_n(
    test_user, ratings_train_relevant, user_train_sequences, ratings_train_by_user,
    svd_model, tfidf_matrix, movies_master, gru_model, gru_movie_to_idx,
    meta_model, scaler, candidate_movie_ids=None, top_n=10, device=device
)
print(top_n_result[["movieId", "title", "svd_score", "content_score", "gru_score", "meta_score"]].to_string())

generating Top-10 for userId=3
   movieId                                                                 title  svd_score  content_score  gru_score  meta_score
0   142115                                                The Blue Planet (2001)   4.813428       0.000000   3.177595    0.882889
1      501                                                          Naked (1993)   4.773522       0.003211  -0.289796    0.874879
2     7099  Nausicaä of the Valley of the Wind (Kaze no tani no Naushika) (1984)   4.651028       0.014393   5.761182    0.871901
3      741                            Ghost in the Shell (Kôkaku kidôtai) (1995)   4.559973       0.023955   4.435933    0.864296
4     1199                                                         Brazil (1985)   4.610057       0.016601   3.504936    0.862864
5     5618                  Spirited Away (Sen to Chihiro no kamikakushi) (2001)   4.637696       0.010550   5.812813    0.862353
6   135456     Ghost in the Shell: Stand Alone Complex - Th

In [14]:
import time
start = time.time()
_ = generate_top_n(test_user, ratings_train_relevant, user_train_sequences, ratings_train_by_user,
                    svd_model, tfidf_matrix, movies_master, gru_model, gru_movie_to_idx,
                    meta_model, scaler, candidate_movie_ids=None, top_n=10, device=device)
print(f"time for one user, full vocab candidates: {time.time()-start:.1f}s")

time for one user, full vocab candidates: 2.3s


In [15]:
# CELL 12 — Wire the sentiment re-ranker onto the real meta-model Top-N output

movie_sentiment = pd.read_parquet(f"{INTERIM_PATH}/movie_sentiment.parquet")  # from Week 3's sentiment work

def rerank_by_sentiment(top_n_movie_ids, movie_sentiment, sentiment_weight=0.3):
    n = len(top_n_movie_ids)
    sentiment_lookup = movie_sentiment.set_index("movieId")[["sentiment_score", "reliable"]].to_dict("index")
    scored = []
    for rank, movie_id in enumerate(top_n_movie_ids):
        original_rank_score = 1.0 - (rank / max(n - 1, 1))
        info = sentiment_lookup.get(movie_id)
        if info is None or not info["reliable"]:
            combined_score = original_rank_score
        else:
            combined_score = (1 - sentiment_weight) * original_rank_score + sentiment_weight * info["sentiment_score"]
        scored.append((movie_id, combined_score))
    scored.sort(key=lambda x: x[1], reverse=True)
    return [m for m, _ in scored]


original_order = top_n_result["movieId"].tolist()
final_order = rerank_by_sentiment(original_order, movie_sentiment, sentiment_weight=0.3)

comparison = pd.DataFrame({
    "original_rank": range(1, len(original_order) + 1),
    "movieId": original_order,
}).merge(movies_master[["movieId", "title"]], on="movieId")
comparison["new_rank"] = comparison["movieId"].apply(lambda m: final_order.index(m) + 1)

print(comparison.sort_values("new_rank")[["new_rank", "original_rank", "title"]].to_string(index=False))

 new_rank  original_rank                                                                title
        1              1                                               The Blue Planet (2001)
        2              2                                                         Naked (1993)
        3              3 Nausicaä of the Valley of the Wind (Kaze no tani no Naushika) (1984)
        4              4                           Ghost in the Shell (Kôkaku kidôtai) (1995)
        5              5                                                        Brazil (1985)
        6              6                 Spirited Away (Sen to Chihiro no kamikakushi) (2001)
        7              7    Ghost in the Shell: Stand Alone Complex - The Laughing Man (2005)
        8              8                             Princess Mononoke (Mononoke-hime) (1997)
        9              9                                                            Moonlight
       10             10                                    

In [16]:
# Diagnostic: check the actual sentiment scores + reliability for these 10 movies
sentiment_lookup = movie_sentiment.set_index("movieId")[["sentiment_score", "review_count", "reliable"]]
check = comparison.merge(sentiment_lookup, on="movieId", how="left")
print(check[["title", "sentiment_score", "review_count", "reliable"]].to_string(index=False))

                                                               title  sentiment_score  review_count reliable
                                              The Blue Planet (2001)              NaN           NaN      NaN
                                                        Naked (1993)              NaN           NaN      NaN
Nausicaä of the Valley of the Wind (Kaze no tani no Naushika) (1984)         0.798274          15.0     True
                          Ghost in the Shell (Kôkaku kidôtai) (1995)              NaN           NaN      NaN
                                                       Brazil (1985)              NaN           NaN      NaN
                Spirited Away (Sen to Chihiro no kamikakushi) (2001)              NaN           NaN      NaN
   Ghost in the Shell: Stand Alone Complex - The Laughing Man (2005)              NaN           NaN      NaN
                            Princess Mononoke (Mononoke-hime) (1997)              NaN           NaN      NaN
                   

In [17]:
# Test the pipeline + re-ranker on 2-3 more users to see if this pattern holds,
# or if other users' recommendations get real sentiment coverage and visible reordering

other_test_users = list(user_train_sequences.keys())[1:4]

for uid in other_test_users:
    top_n = generate_top_n(uid, ratings_train_relevant, user_train_sequences, ratings_train_by_user,
                            svd_model, tfidf_matrix, movies_master, gru_model, gru_movie_to_idx,
                            meta_model, scaler, candidate_movie_ids=None, top_n=10, device=device)
    original = top_n["movieId"].tolist()
    final = rerank_by_sentiment(original, movie_sentiment, sentiment_weight=0.3)

    comp = pd.DataFrame({"original_rank": range(1, 11), "movieId": original}).merge(
        movies_master[["movieId", "title"]], on="movieId"
    ).merge(sentiment_lookup, on="movieId", how="left")
    comp["new_rank"] = comp["movieId"].apply(lambda m: final.index(m) + 1)

    n_reliable = comp["reliable"].sum()
    n_reordered = (comp["original_rank"] != comp["new_rank"]).sum()
    print(f"\n--- userId={uid} --- reliable sentiment coverage: {n_reliable}/10, reordered: {n_reordered}/10")
    print(comp.sort_values("new_rank")[["new_rank", "original_rank", "title", "sentiment_score", "reliable"]].to_string(index=False))


--- userId=84 --- reliable sentiment coverage: 1/10, reordered: 0/10
 new_rank  original_rank                                                             title  sentiment_score reliable
        1              1                                   Decalogue, The (Dekalog) (1989)              NaN      NaN
        2              2                                Come and See (Idi i smotri) (1985)           0.6256     True
        3              3                  Fanny and Alexander (Fanny och Alexander) (1982)              NaN      NaN
        4              4                                               Planet Earth (2006)              NaN      NaN
        5              5                Swedish Love Story, A (Kärlekshistoria, En) (1970)              NaN      NaN
        6              6                                         Harakiri (Seppuku) (1962)              NaN      NaN
        7              7                    Best of Youth, The (La meglio gioventù) (2003)              NaN    

In [20]:
# CELL — Load a small set of real cold-start users for testing (separate from ratings_train_relevant)

cold_start_users_full = set(pd.read_parquet(f"{INTERIM_PATH}/warm_users.parquet")["userId"])  # reload for clarity
ratings_train_full_users = pd.read_parquet(f"{INTERIM_PATH}/ratings_train.parquet", columns=["userId", "movieId", "rating", "timestamp"])
all_train_users_full = set(ratings_train_full_users["userId"].unique())

warm_users_set = set(pd.read_parquet(f"{INTERIM_PATH}/warm_users.parquet")["userId"])
cold_start_users_full = all_train_users_full - warm_users_set
print("genuine cold-start users (full dataset):", len(cold_start_users_full))

# Pull just a handful of cold-start users' train ratings for this test,
# rather than loading everyone's data
test_cold_user = list(cold_start_users_full)[0]
cold_user_ratings = ratings_train_full_users[ratings_train_full_users["userId"] == test_cold_user]
print(f"\ntesting userId={test_cold_user} (train ratings: {len(cold_user_ratings)})")

# Extend ratings_train_relevant with just this one cold-start user's data,
# so get_recommendations() can look them up correctly
ratings_train_relevant_extended = pd.concat([ratings_train_relevant, cold_user_ratings], ignore_index=True)

# Also extend ratings_train_by_user (liked-movies lookup for content scoring)
# in case this user has any rating >= 4.0
cold_user_liked = cold_user_ratings[cold_user_ratings["rating"] >= 4.0]["movieId"].tolist()
if cold_user_liked:
    ratings_train_by_user[test_cold_user] = cold_user_liked

result, path = get_recommendations(
    test_cold_user, ratings_train_relevant_extended, user_train_sequences, ratings_train_by_user,
    svd_model, tfidf_matrix, movies_master, gru_model, gru_movie_to_idx,
    meta_model, scaler, movie_sentiment, popularity_ranking,
    candidate_movie_ids=None, top_n=10, device=device
)
print(f"path taken: {path}")
print(result[["movieId", "title"]].to_string(index=False))

genuine cold-start users (full dataset): 15593

testing userId=65536 (train ratings: 23)
path taken: cold_start
 movieId                                                     title
   59615 Indiana Jones and the Kingdom of the Crystal Skull (2008)
  115210                                               Fury (2014)
    2324                Life Is Beautiful (La Vita è bella) (1997)
    2115               Indiana Jones and the Temple of Doom (1984)
  116797                                 The Imitation Game (2014)
   68157                               Inglourious Basterds (2009)
     912                                         Casablanca (1942)
    8972                                  National Treasure (2004)
    1208                                     Apocalypse Now (1979)
    4310                                       Pearl Harbor (2001)


In [21]:
# CELL — Save meta-model, unified pipeline test results, and Week 3 summary

import pickle, json

# Meta-model + scaler (if not already saved from Cell 10)
with open("/kaggle/working/meta_model.pkl", "wb") as f:
    pickle.dump(meta_model, f)
with open("/kaggle/working/meta_model_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# GBM (kept for reference/comparison, even though logistic regression is final)
with open("/kaggle/working/gbm_comparison_model.pkl", "wb") as f:
    pickle.dump(gbm_model, f)

# Popularity ranking (used by both cold-start fallback and future serving)
with open("/kaggle/working/popularity_ranking.pkl", "wb") as f:
    pickle.dump(popularity_ranking, f)

# Cold-start test example, for documentation/reproducibility
cold_start_test_result = {
    "test_user_id": int(test_cold_user),
    "train_rating_count": int(len(cold_user_ratings)),
    "path_taken": path,
    "top_10_recommendations": result[["movieId", "title"]].to_dict("records"),
}

# Warm-user test examples (from earlier in the session)
warm_user_test_summary = {
    "note": "Tested on 4 warm users (3, 84, 91, 187). Sentiment re-ranking "
            "showed a clear, taste-dependent pattern: mainstream/contemporary-"
            "taste users (e.g. user 91: Titanic, Forrest Gump, Schindler's List) "
            "saw real reordering (4/10 positions changed), while arthouse/"
            "foreign-taste users (e.g. user 84: Bergman, Tarkovsky-adjacent "
            "titles) saw almost none (0-1/10 reliable sentiment coverage), "
            "directly traceable to the RT coverage skew documented in Week 1.",
    "example_users": {
        "user_3": {"reliable_sentiment_coverage": "1/10 (only Nausicaä matched)", "reordered": "0/10"},
        "user_84": {"reliable_sentiment_coverage": "1/10", "reordered": "0/10"},
        "user_91": {"reliable_sentiment_coverage": "7/10", "reordered": "4/10"},
        "user_187": {"reliable_sentiment_coverage": "4/10", "reordered": "2/10"},
    },
}

week3_pipeline_results = {
    "cold_start_threshold": COLD_START_THRESHOLD,
    "meta_model": {
        "type": "Logistic Regression",
        "test_auc": 0.7877,
        "coefficients": {"svd_score": 1.3708, "content_score": 0.0996, "gru_score": 0.0494},
        "gbm_comparison_auc": 0.7929,
        "checkpoint_decision": "PASS -- meta-model retained, fixed-weight ensemble "
                                "fallback (spec Section 5.4 contingency) not needed.",
    },
    "sentiment_reranker": {
        "sentiment_weight": 0.3,
        "min_reliable_reviews": 5,
        "reliable_movie_coverage_pct": 95.4,
    },
    "cold_start_routing_test": cold_start_test_result,
    "warm_user_reranking_test": warm_user_test_summary,
    "pipeline_status": "End-to-end confirmed: cold-start routing -> "
                        "(meta-model OR content+popularity fallback) -> "
                        "sentiment re-rank -> final Top-N. Tested successfully "
                        "on both warm and cold-start real users.",
}

with open("/kaggle/working/week3_pipeline_results.json", "w") as f:
    json.dump(week3_pipeline_results, f, indent=2, default=str)

print(json.dumps(week3_pipeline_results, indent=2, default=str))
print("\nsaved: meta_model.pkl, meta_model_scaler.pkl, gbm_comparison_model.pkl, "
      "popularity_ranking.pkl, week3_pipeline_results.json")

{
  "cold_start_threshold": 25,
  "meta_model": {
    "type": "Logistic Regression",
    "test_auc": 0.7877,
    "coefficients": {
      "svd_score": 1.3708,
      "content_score": 0.0996,
      "gru_score": 0.0494
    },
    "gbm_comparison_auc": 0.7929,
    "checkpoint_decision": "PASS -- meta-model retained, fixed-weight ensemble fallback (spec Section 5.4 contingency) not needed."
  },
  "sentiment_reranker": {
    "sentiment_weight": 0.3,
    "min_reliable_reviews": 5,
    "reliable_movie_coverage_pct": 95.4
  },
  "cold_start_routing_test": {
    "test_user_id": 65536,
    "train_rating_count": 23,
    "path_taken": "cold_start",
    "top_10_recommendations": [
      {
        "movieId": 59615,
        "title": "Indiana Jones and the Kingdom of the Crystal Skull (2008)"
      },
      {
        "movieId": 115210,
        "title": "Fury (2014)"
      },
      {
        "movieId": 2324,
        "title": "Life Is Beautiful (La Vita \u00e8 bella) (1997)"
      },
      {
        "mov